# Reliability-Aware Lightweight Intrusion Detection for IoT and IIoT Edge Communication Networks

Run this notebook from the repository root so that `src/` and `configs/default.yaml` are available. Place CICIoT2023 and Edge-IIoTset under `data/raw/`, or edit the Configuration cell before running experiments.


## Setup


In [ ]:
# Install packages that are not always present in Colab.
!pip -q install lightgbm scikit-learn psutil tqdm joblib pyarrow openpyxl seaborn pyyaml

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

print("Python:", sys.version)
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Runtime check warning:", type(exc).__name__, exc)


In [ ]:
# Optional Google Drive mount and project root.
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
PROJECT_NAME = "ids_edge_reliability"  #@param {type:"string"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    COLAB_DRIVE_MOUNT = Path("/content") / "drive"
    COLAB_DRIVE_ROOT = COLAB_DRIVE_MOUNT / "MyDrive"
    drive.mount(str(COLAB_DRIVE_MOUNT))
    PROJECT_ROOT = COLAB_DRIVE_ROOT / PROJECT_NAME
else:
    PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "configs" / "default.yaml").exists() and (PROJECT_ROOT / "iot" / "configs" / "default.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT / "iot"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)


## Imports


In [ ]:
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit

from src.data.preprocessing import (
    load_prepared_csv,
    load_edgeiiot_dataset,
    make_synthetic_ciciot_demo,
    split_70_15_15,
    split_train_val,
    numeric_feature_columns,
    fit_transform_features,
)
from src.evaluation.metrics import (
    METRIC_COLS,
    RESOURCE_COLS,
    compute_metrics,
    fit_binary_calibrators,
    apply_threshold_policy,
)
from src.models.models import clone_tiny_mlp, apply_structured_pruning, quantize_dynamic_cpu
from src.training.train import (
    get_device,
    memory_report,
    cleanup_memory,
    train_tiny_mlp as train_tiny_mlp_base,
    fit_classic_model as fit_classic_model_base,
    predict_model,
    benchmark_inference,
    benchmark_torch_variant,
    save_model_size,
    save_torch_size,
    model_names_to_run as model_names_to_run_base,
    settings_from_config,
)
from src.utils.config import load_config, resolve_path
from src.utils.io import (
    copy_file_to_local_once,
    merge_csvs_streaming,
    unzip_ciciot_once,
    write_results,
    result_exists as result_exists_base,
)

DEVICE = get_device()
print("DEVICE:", DEVICE)


## Configuration


In [ ]:
# Dataset mode.
RUN_SYNTHETIC_DEMO = False  #@param {type:"boolean"}

CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"
CONFIG = load_config(CONFIG_PATH)

DATA_DIR = resolve_path(CONFIG["paths"]["data_dir"], PROJECT_ROOT)
RAW_DIR = resolve_path(CONFIG["paths"]["raw_dir"], PROJECT_ROOT)
PROCESSED_DIR = resolve_path(CONFIG["paths"]["processed_dir"], PROJECT_ROOT)
OUTPUT_DIR = resolve_path(CONFIG["paths"]["results_dir"], PROJECT_ROOT)
MODEL_DIR = resolve_path(CONFIG["paths"]["models_dir"], PROJECT_ROOT)
FIG_DIR = resolve_path(CONFIG["paths"]["figures_dir"], PROJECT_ROOT)

for path in [DATA_DIR, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATASETS_ROOT = str(RAW_DIR)  #@param {type:"string"}
DATASETS_ROOT_PATH = Path(DATASETS_ROOT)

USE_CICIOT_ZIP = False  #@param {type:"boolean"}
CICIOT_ZIP_PATH = str(resolve_path(CONFIG["datasets"]["ciciot"]["zip_path"], PROJECT_ROOT))  #@param {type:"string"}
CICIOT_EXTRACT_DIR = str(resolve_path(CONFIG["datasets"]["ciciot"]["extract_dir"], PROJECT_ROOT))  #@param {type:"string"}
CICIOT_MERGED_CSV = str(resolve_path(CONFIG["datasets"]["ciciot"]["merged_csv"], PROJECT_ROOT))

COLAB_TMP_DIR = Path("/content") if Path("/content").exists() else PROJECT_ROOT
COPY_CICIOT_TO_LOCAL = Path("/content").exists()  #@param {type:"boolean"}
LOCAL_CICIOT_CSV = str(COLAB_TMP_DIR / "CICIoT2023.csv")  #@param {type:"string"}
CICIOT_CSV_PATH = CICIOT_MERGED_CSV  #@param {type:"string"}

MERGE_CICIOT_FROM_DIR = False  #@param {type:"boolean"}
CICIOT_RAW_CSV_DIR = str(resolve_path(CONFIG["datasets"]["ciciot"]["raw_csv_dir"], PROJECT_ROOT))  #@param {type:"string"}
MERGED_CICIOT_OUTPUT = CICIOT_MERGED_CSV
MAX_ROWS_PER_FILE_WHEN_MERGING = 0  #@param {type:"integer"}

EDGEIIOT_CSV_PATH = str(resolve_path(CONFIG["datasets"]["edgeiiot"]["csv_path"], PROJECT_ROOT))  #@param {type:"string"}
COPY_EDGEIIOT_TO_LOCAL = Path("/content").exists()  #@param {type:"boolean"}
LOCAL_EDGEIIOT_CSV = str(COLAB_TMP_DIR / "DNN-EdgeIIoT-dataset.csv")  #@param {type:"string"}

EDGE_DROP_DUPLICATES = bool(CONFIG["datasets"]["edgeiiot"].get("drop_duplicates", True))  #@param {type:"boolean"}
EDGE_USE_AUTHOR_STYLE_DROPS = bool(CONFIG["datasets"]["edgeiiot"].get("use_author_style_drops", True))  #@param {type:"boolean"}

FAST_MODE = bool(CONFIG["training"].get("fast_mode", False))  #@param {type:"boolean"}
SAMPLE_ROWS_CICIOT = int(CONFIG["datasets"]["ciciot"].get("sample_rows", 1000000))  #@param {type:"integer"}
SAMPLE_ROWS_EDGEIIOT = int(CONFIG["datasets"]["edgeiiot"].get("sample_rows", 1000000))  #@param {type:"integer"}
E2_MAX_TRAIN_ROWS = int(CONFIG["training"].get("e2_max_train_rows", 250000))  #@param {type:"integer"}
RANDOM_SEEDS = list(CONFIG.get("random_seeds", [13, 21, 34, 42, 55]))
SKIP_IF_RESULT_EXISTS = bool(CONFIG["training"].get("skip_if_result_exists", True))  #@param {type:"boolean"}

RUN_LR = bool(CONFIG["models"].get("run_lr", True))
RUN_RF = bool(CONFIG["models"].get("run_rf", True))
RUN_LGBM = bool(CONFIG["models"].get("run_lgbm", True))
RUN_TINYMLP = bool(CONFIG["models"].get("run_tinymlp", True))

TRAINING_SETTINGS = settings_from_config(CONFIG)
TRAINING_SETTINGS.fast_mode = FAST_MODE
TRAINING_SETTINGS.run_lr = RUN_LR
TRAINING_SETTINGS.run_rf = RUN_RF
TRAINING_SETTINGS.run_lgbm = RUN_LGBM
TRAINING_SETTINGS.run_tinymlp = RUN_TINYMLP

TINYMLP_EPOCHS = TRAINING_SETTINGS.effective_tinymlp_epochs
TINYMLP_BATCH_SIZE = TRAINING_SETTINGS.tinymlp_batch_size
RF_N_ESTIMATORS = TRAINING_SETTINGS.effective_rf_n_estimators
RF_MAX_DEPTH = TRAINING_SETTINGS.effective_rf_max_depth
LGBM_N_ESTIMATORS = TRAINING_SETTINGS.effective_lgbm_n_estimators

RUN_EXPERIMENTS = False  #@param {type:"boolean"}
RUN_E1_BASELINES = RUN_EXPERIMENTS
RUN_E2_ZERO_DAY = RUN_EXPERIMENTS
RUN_E3_COMPRESSION = RUN_EXPERIMENTS
RUN_E4_FEATURE_ABLATION = RUN_EXPERIMENTS
RUN_E5_CALIBRATION = RUN_EXPERIMENTS
RUN_E6_EDGE_BASELINES = RUN_EXPERIMENTS
RUN_E7_EDGE_CALIBRATION = RUN_EXPERIMENTS
RUN_E8_EDGE_COMPRESSION = RUN_EXPERIMENTS

def result_exists(path):
    return result_exists_base(path, skip_if_exists=SKIP_IF_RESULT_EXISTS)

def train_tiny_mlp(X_train, y_train, X_val, y_val, num_classes, seed, epochs=None):
    return train_tiny_mlp_base(
        X_train,
        y_train,
        X_val,
        y_val,
        num_classes,
        seed,
        epochs=epochs,
        settings=TRAINING_SETTINGS,
        device=DEVICE,
    )

def fit_classic_model(model_name, X_train, y_train, seed):
    return fit_classic_model_base(model_name, X_train, y_train, seed, settings=TRAINING_SETTINGS)

def model_names_to_run():
    return model_names_to_run_base(TRAINING_SETTINGS)

print(json.dumps({
    "RUN_SYNTHETIC_DEMO": RUN_SYNTHETIC_DEMO,
    "DATASETS_ROOT": DATASETS_ROOT,
    "CICIOT_CSV_PATH": CICIOT_CSV_PATH,
    "CICIOT_EXTRACT_DIR": CICIOT_EXTRACT_DIR,
    "COPY_CICIOT_TO_LOCAL": COPY_CICIOT_TO_LOCAL,
    "LOCAL_CICIOT_CSV": LOCAL_CICIOT_CSV,
    "EDGEIIOT_CSV_PATH": EDGEIIOT_CSV_PATH,
    "COPY_EDGEIIOT_TO_LOCAL": COPY_EDGEIIOT_TO_LOCAL,
    "LOCAL_EDGEIIOT_CSV": LOCAL_EDGEIIOT_CSV,
    "FAST_MODE": FAST_MODE,
    "SAMPLE_ROWS_CICIOT": SAMPLE_ROWS_CICIOT,
    "SAMPLE_ROWS_EDGEIIOT": SAMPLE_ROWS_EDGEIIOT,
    "TINYMLP_EPOCHS": TINYMLP_EPOCHS,
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "RUN_EXPERIMENTS": RUN_EXPERIMENTS
}, indent=2))


For a quick check of the notebook mechanics, set `RUN_SYNTHETIC_DEMO = True` and keep `RUN_EXPERIMENTS = False` until the dataset paths are confirmed. For the reported experiment design, use the configured CICIoT2023 and Edge-IIoTset CSV files, set the desired experiment switches, and keep `RANDOM_SEEDS = [13, 21, 34, 42, 55]`.


## Data loading


In [ ]:
def copy_ciciot_to_local_once(source_csv, local_csv):
    return copy_file_to_local_once(source_csv, local_csv, enabled=COPY_CICIOT_TO_LOCAL, label="CICIoT CSV")

if RUN_SYNTHETIC_DEMO:
    CICIOT_CSV_PATH = make_synthetic_ciciot_demo(PROCESSED_DIR / "ciciot2023_synthetic_demo.csv")
elif USE_CICIOT_ZIP:
    raw_csv_dir = unzip_ciciot_once(CICIOT_ZIP_PATH, CICIOT_EXTRACT_DIR)
    merged_csv = merge_csvs_streaming(
        raw_csv_dir,
        CICIOT_MERGED_CSV,
        max_rows_per_file=None if MAX_ROWS_PER_FILE_WHEN_MERGING <= 0 else MAX_ROWS_PER_FILE_WHEN_MERGING,
        skip_if_exists=SKIP_IF_RESULT_EXISTS,
    )
    CICIOT_CSV_PATH = copy_ciciot_to_local_once(merged_csv, LOCAL_CICIOT_CSV)
elif MERGE_CICIOT_FROM_DIR:
    merged_csv = merge_csvs_streaming(
        CICIOT_RAW_CSV_DIR,
        MERGED_CICIOT_OUTPUT,
        max_rows_per_file=None if MAX_ROWS_PER_FILE_WHEN_MERGING <= 0 else MAX_ROWS_PER_FILE_WHEN_MERGING,
        skip_if_exists=SKIP_IF_RESULT_EXISTS,
    )
    CICIOT_CSV_PATH = copy_ciciot_to_local_once(merged_csv, LOCAL_CICIOT_CSV)
else:
    CICIOT_CSV_PATH = copy_ciciot_to_local_once(CICIOT_CSV_PATH, LOCAL_CICIOT_CSV)
print("Active CICIoT CSV:", CICIOT_CSV_PATH)

EDGEIIOT_ACTIVE_CSV = ""
if EDGEIIOT_CSV_PATH:
    try:
        EDGEIIOT_ACTIVE_CSV = copy_file_to_local_once(
            EDGEIIOT_CSV_PATH,
            LOCAL_EDGEIIOT_CSV,
            enabled=COPY_EDGEIIOT_TO_LOCAL,
            label="Edge-IIoTset CSV",
        )
    except Exception as exc:
        print("[WARN] Edge-IIoTset path not ready:", type(exc).__name__, exc)
print("Active Edge-IIoTset CSV:", EDGEIIOT_ACTIVE_CSV if EDGEIIOT_ACTIVE_CSV else "not configured")


## Preprocessing


Reusable preprocessing, metric, model, and training utilities are imported from `src/`.


In [ ]:
if not CICIOT_CSV_PATH:
    raise ValueError("Set CICIOT_CSV_PATH, enable RUN_SYNTHETIC_DEMO, or enable MERGE_CICIOT_FROM_DIR.")

ciciot_df, ciciot_label_col = load_prepared_csv(
    CICIOT_CSV_PATH,
    sample_rows=SAMPLE_ROWS_CICIOT if SAMPLE_ROWS_CICIOT > 0 else None,
    seed=42,
    dataset_name="CICIoT2023-main"
)
memory_report()
display(ciciot_df.head())


## Training


In [ ]:
# E1 - In-dataset binary and multiclass baselines
def run_e1_baselines(df, label_col, out_dir):
    out_path = Path(out_dir) / "E1_baselines.csv"
    if result_exists(out_path):
        print("Skipping E1:", out_path)
        return pd.read_csv(out_path)
    rows = []
    for task in ["binary", "multiclass"]:
        if task == "multiclass":
            task_df = df[df["Category"] != "Benign"].copy().reset_index(drop=True)
            if task_df["Category"].nunique() < 2:
                continue
            y_split = task_df["Category"].astype(str).to_numpy()
        else:
            task_df = df.copy().reset_index(drop=True)
            y_split = task_df["BinaryLabel"].to_numpy()
            if len(np.unique(y_split)) < 2:
                continue
        for seed in RANDOM_SEEDS:
            print(f"\n[E1] task={task} seed={seed}")
            train_idx, val_idx, test_idx = split_70_15_15(y_split, seed)
            X_train, y_train, X_val, y_val, X_test, y_test, enc, sc, med, fcols = fit_transform_features(task_df, label_col, task, train_idx, val_idx, test_idx)
            for model_name in model_names_to_run():
                try:
                    print("[E1] training", model_name)
                    t0 = time.perf_counter()
                    model = train_tiny_mlp(X_train, y_train, X_val, y_val, len(np.unique(y_train)), seed) if model_name == "tinymlp" else fit_classic_model(model_name, X_train, y_train, seed)
                    train_sec = time.perf_counter() - t0
                    pred, probs = predict_model(model_name, model, X_test)
                    row = {"experiment":"E1", "task":task, "seed":seed, "model":model_name, "train_sec":train_sec, "n_features":len(fcols)}
                    row.update(compute_metrics(y_test, pred, probs, task))
                    model_path = MODEL_DIR / "E1" / (f"{task}_{model_name}_seed{seed}.pt" if model_name=="tinymlp" else f"{task}_{model_name}_seed{seed}.joblib")
                    row["model_size_mb"] = save_model_size(model_name, model, model_path)
                    row.update(benchmark_inference(model_name, model, X_test))
                    rows.append(row)
                    print({k: row[k] for k in ["macro_f1","mcc","auprc","auroc","ece","fa_10k"] if k in row})
                except Exception as exc:
                    print(f"[E1][WARN] {model_name} failed:", type(exc).__name__, exc)
                cleanup_memory()
            del X_train, X_val, X_test, y_train, y_val, y_test
            cleanup_memory()
    return write_results(rows, out_path, ["experiment","task","seed","model","train_sec","n_features"])

if RUN_E1_BASELINES:
    e1_df = run_e1_baselines(ciciot_df, ciciot_label_col, OUTPUT_DIR)
    display(e1_df.sort_values(["task","macro_f1"], ascending=[True, False]).head(30))


In [ ]:
# E2 - Zero-day / leave-one-attack-category-out evaluation
def holdout_split_zero_day(df, heldout_cat, seed):
    benign_idx = df.index[df["Category"] == "Benign"].to_numpy()
    rng = np.random.default_rng(seed); rng.shuffle(benign_idx)
    split = int(0.8 * len(benign_idx))
    benign_train, benign_test = set(benign_idx[:split]), set(benign_idx[split:])
    train_idx = set(df.index[(df["Category"] != heldout_cat) & (df["Category"] != "Benign")].to_list()) | benign_train
    test_idx = set(df.index[df["Category"] == heldout_cat].to_list()) | benign_test
    return np.array(sorted(train_idx)), np.array(sorted(test_idx))

def run_e2_zero_day(df, label_col, out_dir):
    out_path = Path(out_dir) / "E2_zero_day.csv"
    if result_exists(out_path):
        print("Skipping E2:", out_path)
        return pd.read_csv(out_path)
    rows = []
    feature_cols = numeric_feature_columns(df, label_col)
    categories = [c for c in sorted(df["Category"].astype(str).unique()) if c not in ["Benign","Unknown"]]
    for heldout in categories:
        for seed in RANDOM_SEEDS:
            print(f"\n[E2] heldout={heldout} seed={seed}")
            train_idx, test_idx = holdout_split_zero_day(df, heldout, seed)
            y_train_full = (df.iloc[train_idx]["Category"] != "Benign").astype(int).to_numpy()
            if len(train_idx) > E2_MAX_TRAIN_ROWS:
                sp = StratifiedShuffleSplit(n_splits=1, train_size=E2_MAX_TRAIN_ROWS, random_state=seed)
                sub, _ = next(sp.split(np.zeros(len(train_idx)), y_train_full))
                train_idx = train_idx[sub]; y_train_full = y_train_full[sub]
            tr_rel, va_rel = split_train_val(y_train_full, seed+7)
            inner_train_idx, inner_val_idx = train_idx[tr_rel], train_idx[va_rel]
            X_train, y_train, X_val, y_val, X_test, y_test, enc, sc, med, fcols = fit_transform_features(df, label_col, "binary", inner_train_idx, inner_val_idx, test_idx, feature_cols)
            for model_name in model_names_to_run():
                try:
                    model = train_tiny_mlp(X_train, y_train, X_val, y_val, 2, seed) if model_name=="tinymlp" else fit_classic_model(model_name, X_train, y_train, seed)
                    pred, probs = predict_model(model_name, model, X_test)
                    row = {"experiment":"E2", "protocol":"leave_one_category_out", "heldout":heldout, "seed":seed, "model":model_name, "train_size":len(X_train), "test_size":len(X_test)}
                    row.update(compute_metrics(y_test, pred, probs, "binary"))
                    rows.append(row)
                    print(model_name, {k: row[k] for k in ["macro_f1","malicious_recall","mcc","auprc","fa_10k"]})
                except Exception as exc:
                    print(f"[E2][WARN] {model_name} failed:", type(exc).__name__, exc)
                cleanup_memory()
            del X_train, X_val, X_test, y_train, y_val, y_test
            cleanup_memory()
    return write_results(rows, out_path, ["experiment","protocol","heldout","seed","model","train_size","test_size"])

if RUN_E2_ZERO_DAY:
    e2_df = run_e2_zero_day(ciciot_df, ciciot_label_col, OUTPUT_DIR)
    display(e2_df.groupby(["heldout","model"])[METRIC_COLS].mean(numeric_only=True).reset_index().sort_values(["heldout","macro_f1"], ascending=[True,False]).head(80))


In [ ]:
# E3 - Compression: FP32 TinyMLP, pruning, INT8 dynamic quantization
def run_e3_compression(df, label_col, out_dir):
    out_path = Path(out_dir) / "E3_compression.csv"
    if result_exists(out_path):
        print("Skipping E3:", out_path)
        return pd.read_csv(out_path)
    rows = []
    y = df["BinaryLabel"].to_numpy()
    fcols = numeric_feature_columns(df, label_col)
    for seed in RANDOM_SEEDS:
        print(f"\n[E3] seed={seed}")
        train_idx, val_idx, test_idx = split_70_15_15(y, seed)
        X_train, y_train, X_val, y_val, X_test, y_test, enc, sc, med, used = fit_transform_features(df, label_col, "binary", train_idx, val_idx, test_idx, fcols)
        fp32 = train_tiny_mlp(X_train, y_train, X_val, y_val, 2, seed)
        variants = {
            "fp32_gpu": (fp32.to(DEVICE).eval(), False),
            "pruned30_fp32": (apply_structured_pruning(clone_tiny_mlp(fp32, X_train.shape[1], 2), 0.30).to(DEVICE).eval(), False),
            "int8_dynamic_cpu": (quantize_dynamic_cpu(clone_tiny_mlp(fp32, X_train.shape[1], 2)), True),
            "pruned30_int8_cpu": (quantize_dynamic_cpu(apply_structured_pruning(clone_tiny_mlp(fp32, X_train.shape[1], 2), 0.30)), True),
        }
        for variant, (model, quantized) in variants.items():
            try:
                pred, probs, res = benchmark_torch_variant(model, X_test, quantized=quantized)
                row = {"experiment":"E3", "seed":seed, "model":"tinymlp", "variant":variant, "model_size_mb": save_torch_size(model, MODEL_DIR/"E3"/f"{variant}_seed{seed}.pt"), "n_features":len(used)}
                row.update(res); row.update(compute_metrics(y_test, pred, probs, "binary"))
                rows.append(row)
                print(variant, {k: row[k] for k in ["macro_f1","model_size_mb","latency_per_flow_sec","throughput_flows_sec"]})
            except Exception as exc:
                print(f"[E3][WARN] {variant} failed:", type(exc).__name__, exc)
            cleanup_memory()
        del X_train, X_val, X_test, y_train, y_val, y_test, fp32
        cleanup_memory()
    return write_results(rows, out_path, ["experiment","seed","model","variant","n_features"])

if RUN_E3_COMPRESSION:
    e3_df = run_e3_compression(ciciot_df, ciciot_label_col, OUTPUT_DIR)
    display(e3_df.groupby("variant")[["macro_f1","mcc","model_size_mb","latency_per_flow_sec","throughput_flows_sec"]].mean(numeric_only=True).reset_index())


In [ ]:
# E4 - Feature ablation
def infer_feature_groups(cols):
    groups = {"temporal": [], "packet_count": [], "byte_length": [], "rate": [], "tcp_flags_header": [], "all": list(cols)}
    for c in cols:
        cl = c.lower()
        if any(k in cl for k in ["duration","iat","inter","active","idle"]): groups["temporal"].append(c)
        if any(k in cl for k in ["pkt","pkts","packet","packets"]): groups["packet_count"].append(c)
        if any(k in cl for k in ["byte","bytes","byts","len","length","size"]): groups["byte_length"].append(c)
        if any(k in cl for k in ["rate","per_sec","sec"]): groups["rate"].append(c)
        if any(k in cl for k in ["flag","header","syn","ack","fin","rst","psh","urg"]): groups["tcp_flags_header"].append(c)
    return {k: sorted(set(v)) for k, v in groups.items() if len(set(v)) > 0}

def run_e4_feature_ablation(df, label_col, out_dir):
    out_path = Path(out_dir) / "E4_feature_ablation.csv"
    if result_exists(out_path):
        print("Skipping E4:", out_path)
        return pd.read_csv(out_path)
    rows = []
    groups = infer_feature_groups(numeric_feature_columns(df, label_col))
    print("Feature groups:", {k: len(v) for k,v in groups.items()})
    y = df["BinaryLabel"].to_numpy()
    for seed in RANDOM_SEEDS:
        train_idx, val_idx, test_idx = split_70_15_15(y, seed)
        for group, fcols in groups.items():
            if group != "all" and len(fcols) < 2:
                continue
            print(f"\n[E4] group={group} seed={seed} n_features={len(fcols)}")
            try:
                X_train, y_train, X_val, y_val, X_test, y_test, enc, sc, med, used = fit_transform_features(df, label_col, "binary", train_idx, val_idx, test_idx, fcols)
                model_name = "lgbm" if RUN_LGBM else "tinymlp"
                model = fit_classic_model("lgbm", X_train, y_train, seed) if model_name=="lgbm" else train_tiny_mlp(X_train, y_train, X_val, y_val, 2, seed)
                pred, probs = predict_model(model_name, model, X_test)
                row = {"experiment":"E4", "seed":seed, "feature_group":group, "model":model_name, "n_features":len(used)}
                row.update(compute_metrics(y_test, pred, probs, "binary"))
                rows.append(row)
            except Exception as exc:
                print(f"[E4][WARN] {group} failed:", type(exc).__name__, exc)
            cleanup_memory()
    return write_results(rows, out_path, ["experiment","seed","feature_group","model","n_features"])

if RUN_E4_FEATURE_ABLATION:
    e4_df = run_e4_feature_ablation(ciciot_df, ciciot_label_col, OUTPUT_DIR)
    display(e4_df.groupby(["feature_group","model"])[["macro_f1","mcc","auprc","fa_10k","n_features"]].mean(numeric_only=True).reset_index().sort_values("macro_f1", ascending=False))


## Evaluation


In [ ]:
# E5 - Calibration and confidence-threshold policies
def run_e5_calibration(df, label_col, out_dir):
    out_path = Path(out_dir) / "E5_calibration.csv"
    if result_exists(out_path):
        cached_df = pd.read_csv(out_path)
        if "lr" not in cached_df["model"].unique() and RUN_LR:
            print("Rerunning E5: 'lr' is missing from cached results.")
        else:
            print("Skipping E5:", out_path)
            return cached_df

    rows = []
    y = df["BinaryLabel"].to_numpy()
    fcols = numeric_feature_columns(df, label_col)
    calibration_models = model_names_to_run()
    for seed in RANDOM_SEEDS:
        train_idx, val_idx, test_idx = split_70_15_15(y, seed)
        X_train, y_train, X_val, y_val, X_test, y_test, enc, sc, med, used = fit_transform_features(df, label_col, "binary", train_idx, val_idx, test_idx, fcols)
        for model_name in calibration_models:
            try:
                print(f"\n[E5] model={model_name} seed={seed}")
                model = train_tiny_mlp(X_train, y_train, X_val, y_val, 2, seed) if model_name=="tinymlp" else fit_classic_model(model_name, X_train, y_train, seed)
                _, probs_val = predict_model(model_name, model, X_val)
                _, probs_test = predict_model(model_name, model, X_test)
                calibrators = fit_binary_calibrators(probs_val, y_val)
                for cname, fn in calibrators.items():
                    probs = fn(probs_test)
                    pred = probs.argmax(axis=1)
                    row = {"experiment":"E5", "seed":seed, "model":model_name, "calibration":cname, "threshold":np.nan, "policy":"none"}
                    row.update(compute_metrics(y_test, pred, probs, "binary"))
                    rows.append(row)
                    for th in [0.50, 0.60, 0.70, 0.80, 0.90]:
                        pred_th, coverage = apply_threshold_policy(probs, th)
                        th_row = {"experiment":"E5", "seed":seed, "model":model_name, "calibration":cname, "threshold":th, "policy":"conservative_no_alert", "coverage":coverage}
                        th_row.update(compute_metrics(y_test, pred_th, probs, "binary"))
                        rows.append(th_row)
            except Exception as exc:
                print(f"[E5][WARN] {model_name} failed:", type(exc).__name__, exc)
            cleanup_memory()
        del X_train, X_val, X_test, y_train, y_val, y_test
        cleanup_memory()
    return write_results(rows, out_path, ["experiment","seed","model","calibration","threshold","policy","coverage"])

if RUN_E5_CALIBRATION:
    e5_df = run_e5_calibration(ciciot_df, ciciot_label_col, OUTPUT_DIR)
    display(e5_df[e5_df["policy"].eq("none")].groupby(["model","calibration"])[["macro_f1","mcc","ece","brier_score","fa_10k"]].mean(numeric_only=True).reset_index().sort_values(["model","ece"]))


In [ ]:
# E6-E8 - Edge-IIoTset independent validation experiments
edge_df = None
edge_label_col = None

def run_edge_e6_baselines(df, label_col, out_dir):
    out_path = Path(out_dir) / "E6_edgeiiot_baselines.csv"
    if result_exists(out_path):
        print("Skipping E6 Edge baselines:", out_path)
        return pd.read_csv(out_path)
    rows = []
    for task in ["binary", "multiclass"]:
        if task == "multiclass":
            task_df = df[df["Category"] != "Benign"].copy().reset_index(drop=True)
            if task_df["Category"].nunique() < 2:
                print("[E6] Edge multiclass skipped: fewer than two attack classes.")
                continue
            y_split = task_df["Category"].astype(str).to_numpy()
        else:
            task_df = df.copy().reset_index(drop=True)
            y_split = task_df["BinaryLabel"].to_numpy()
            if len(np.unique(y_split)) < 2:
                print("[E6] Edge binary skipped: fewer than two classes.")
                continue
        for seed in RANDOM_SEEDS:
            print(f"\n[E6 Edge] task={task} seed={seed}")
            train_idx, val_idx, test_idx = split_70_15_15(y_split, seed)
            X_train, y_train, X_val, y_val, X_test, y_test, enc, sc, med, used = fit_transform_features(task_df, label_col, task, train_idx, val_idx, test_idx)
            output_dim = len(np.unique(y_train))
            for model_name in model_names_to_run():
                t0 = time.perf_counter()
                try:
                    print("[E6 Edge] training", model_name)
                    model = train_tiny_mlp(X_train, y_train, X_val, y_val, output_dim, seed) if model_name == "tinymlp" else fit_classic_model(model_name, X_train, y_train, seed)
                    pred, probs = predict_model(model_name, model, X_test)
                    res = benchmark_inference(model_name, model, X_test)
                    row = {
                        "experiment":"E6", "dataset":"Edge-IIoTset", "task":task, "seed":seed,
                        "model":model_name, "train_sec":time.perf_counter()-t0,
                        "n_features":len(used), "n_classes":output_dim,
                    }
                    row.update(compute_metrics(y_test, pred, probs, task))
                    row.update(res)
                    try:
                        row["model_size_mb"] = save_model_size(model_name, model, MODEL_DIR/"E6_edge"/(f"{task}_{model_name}_seed{seed}.pt" if model_name=="tinymlp" else f"{task}_{model_name}_seed{seed}.joblib"))
                    except Exception:
                        row["model_size_mb"] = np.nan
                    rows.append(row)
                    print({k: row[k] for k in ["macro_f1", "mcc", "auprc", "auroc", "ece", "fa_10k"] if k in row})
                except Exception as exc:
                    print(f"[E6 Edge][WARN] {task} {model_name} failed:", type(exc).__name__, exc)
                cleanup_memory()
            del X_train, X_val, X_test, y_train, y_val, y_test
            cleanup_memory()
    return write_results(rows, out_path, ["experiment","dataset","task","seed","model","train_sec","n_features","n_classes"])


def run_edge_e7_calibration(df, label_col, out_dir):
    out_path = Path(out_dir) / "E7_edgeiiot_calibration.csv"
    if result_exists(out_path):
        print("Skipping E7 Edge calibration:", out_path)
        return pd.read_csv(out_path)
    rows = []
    y = df["BinaryLabel"].to_numpy()
    fcols = numeric_feature_columns(df, label_col)
    for seed in RANDOM_SEEDS:
        print(f"\n[E7 Edge] seed={seed}")
        train_idx, val_idx, test_idx = split_70_15_15(y, seed)
        X_train, y_train, X_val, y_val, X_test, y_test, enc, sc, med, used = fit_transform_features(df, label_col, "binary", train_idx, val_idx, test_idx, fcols)
        for model_name in model_names_to_run():
            try:
                print("[E7 Edge] calibration for", model_name)
                model = train_tiny_mlp(X_train, y_train, X_val, y_val, 2, seed) if model_name == "tinymlp" else fit_classic_model(model_name, X_train, y_train, seed)
                _, probs_val = predict_model(model_name, model, X_val)
                _, probs_test = predict_model(model_name, model, X_test)
                calibrators = fit_binary_calibrators(probs_val, y_val)
                for cname, fn in calibrators.items():
                    probs = fn(probs_test)
                    pred = probs.argmax(axis=1)
                    row = {"experiment":"E7", "dataset":"Edge-IIoTset", "seed":seed, "model":model_name, "calibration":cname, "threshold":np.nan, "policy":"none", "n_features":len(used)}
                    row.update(compute_metrics(y_test, pred, probs, "binary"))
                    rows.append(row)
                    for th in [0.50, 0.60, 0.70, 0.80, 0.90]:
                        pred_th, coverage = apply_threshold_policy(probs, th)
                        th_row = {"experiment":"E7", "dataset":"Edge-IIoTset", "seed":seed, "model":model_name, "calibration":cname, "threshold":th, "policy":"conservative_no_alert", "coverage":coverage, "n_features":len(used)}
                        th_row.update(compute_metrics(y_test, pred_th, probs, "binary"))
                        rows.append(th_row)
            except Exception as exc:
                print(f"[E7 Edge][WARN] {model_name} failed:", type(exc).__name__, exc)
            cleanup_memory()
        del X_train, X_val, X_test, y_train, y_val, y_test
        cleanup_memory()
    return write_results(rows, out_path, ["experiment","dataset","seed","model","calibration","threshold","policy","coverage","n_features"])


def run_edge_e8_compression(df, label_col, out_dir):
    out_path = Path(out_dir) / "E8_edgeiiot_compression.csv"
    if result_exists(out_path):
        print("Skipping E8 Edge compression:", out_path)
        return pd.read_csv(out_path)
    rows = []
    y = df["BinaryLabel"].to_numpy()
    fcols = numeric_feature_columns(df, label_col)
    for seed in RANDOM_SEEDS:
        print(f"\n[E8 Edge] seed={seed}")
        train_idx, val_idx, test_idx = split_70_15_15(y, seed)
        X_train, y_train, X_val, y_val, X_test, y_test, enc, sc, med, used = fit_transform_features(df, label_col, "binary", train_idx, val_idx, test_idx, fcols)
        fp32 = train_tiny_mlp(X_train, y_train, X_val, y_val, 2, seed)
        variants = {
            "fp32_gpu": (fp32.to(DEVICE).eval(), False),
            "pruned30_fp32": (apply_structured_pruning(clone_tiny_mlp(fp32, X_train.shape[1], 2), 0.30).to(DEVICE).eval(), False),
            "int8_dynamic_cpu": (quantize_dynamic_cpu(clone_tiny_mlp(fp32, X_train.shape[1], 2)), True),
            "pruned30_int8_cpu": (quantize_dynamic_cpu(apply_structured_pruning(clone_tiny_mlp(fp32, X_train.shape[1], 2), 0.30)), True),
        }
        for variant, (model, quantized) in variants.items():
            try:
                pred, probs, res = benchmark_torch_variant(model, X_test, quantized=quantized)
                row = {"experiment":"E8", "dataset":"Edge-IIoTset", "seed":seed, "model":"tinymlp", "variant":variant, "model_size_mb": save_torch_size(model, MODEL_DIR/"E8_edge"/f"{variant}_seed{seed}.pt"), "n_features":len(used)}
                row.update(res)
                row.update(compute_metrics(y_test, pred, probs, "binary"))
                rows.append(row)
                print(variant, {k: row[k] for k in ["macro_f1", "model_size_mb", "latency_per_flow_sec", "throughput_flows_sec"]})
            except Exception as exc:
                print(f"[E8 Edge][WARN] {variant} failed:", type(exc).__name__, exc)
            cleanup_memory()
        del X_train, X_val, X_test, y_train, y_val, y_test, fp32
        cleanup_memory()
    return write_results(rows, out_path, ["experiment","dataset","seed","model","variant","n_features"])



# Load and run Edge-IIoTset independent validation.
edge_df, edge_label_col = None, None


def resolve_edgeiiot_active_csv():
    """Resolve the configured Edge-IIoTset CSV if the preparation cell was skipped."""
    from pathlib import Path

    # Use existing active path if available and valid.
    existing = globals().get("EDGEIIOT_ACTIVE_CSV", "")
    if existing and Path(existing).exists():
        return existing

    data_root = globals().get("DATASETS_ROOT", str(Path("/content") / PROJECT_NAME / "data" / "raw"))
    configured = globals().get("EDGEIIOT_CSV_PATH", "")
    local_path = globals().get("LOCAL_EDGEIIOT_CSV", "/content/DNN-EdgeIIoT-dataset.csv")
    copy_enabled = bool(globals().get("COPY_EDGEIIOT_TO_LOCAL", True))

    candidates = []
    if local_path:
        candidates.append(local_path)
    if configured:
        candidates.append(configured)
    candidates.append(f"{data_root}/Edge-IIoTset/DNN-EdgeIIoT-dataset.csv")

    # First prefer local copy if it already exists.
    for cand in candidates:
        if cand and str(cand).startswith("/content/") and not str(cand).startswith("/content/drive") and Path(cand).exists():
            print("Using existing local Edge-IIoTset CSV:", cand)
            return cand

    # Then find configured file and copy it to local if requested.
    for cand in candidates:
        p = Path(cand)
        if p.exists():
            if copy_enabled and str(p).startswith("/content/drive"):
                try:
                    return copy_file_to_local_once(str(p), local_path, enabled=True, label="Edge-IIoTset CSV")
                except NameError:
                    # Minimal fallback if helper was not executed.
                    import shutil
                    lp = Path(local_path)
                    if lp.exists() and lp.stat().st_size == p.stat().st_size:
                        print("Local Edge-IIoTset CSV already exists and matches source size:", lp)
                        return str(lp)
                    print(f"Copying Edge-IIoTset CSV:\n  {p}\n  -> {lp}")
                    shutil.copy2(p, lp)
                    return str(lp)
                except Exception as exc:
                    print("[WARN] Could not copy Edge-IIoTset CSV to local; using configured path instead:", type(exc).__name__, exc)
                    return str(p)
            print("Using Edge-IIoTset CSV:", p)
            return str(p)

    raise ValueError(
        "Edge-IIoTset CSV not found. Checked:\n" + "\n".join(str(c) for c in candidates)
    )

if any([RUN_E6_EDGE_BASELINES, RUN_E7_EDGE_CALIBRATION, RUN_E8_EDGE_COMPRESSION]):
    EDGEIIOT_ACTIVE_CSV = resolve_edgeiiot_active_csv()
    print("Active Edge-IIoTset CSV:", EDGEIIOT_ACTIVE_CSV)
    edge_df, edge_label_col = load_edgeiiot_dataset(
        EDGEIIOT_ACTIVE_CSV,
        sample_rows=SAMPLE_ROWS_EDGEIIOT if SAMPLE_ROWS_EDGEIIOT > 0 else None,
        seed=42,
        drop_duplicates=EDGE_DROP_DUPLICATES,
        use_author_style_drops=EDGE_USE_AUTHOR_STYLE_DROPS,
    )
    memory_report()
    display(edge_df.head())
else:
    print("Skipping Edge-IIoTset independent validation because no experiments were run.")

if RUN_E6_EDGE_BASELINES and edge_df is not None:
    e6_edge_df = run_edge_e6_baselines(edge_df, edge_label_col, OUTPUT_DIR)
    display(e6_edge_df.groupby(["task","model"])[["macro_f1","mcc","auprc","auroc","ece","fa_10k","n_features"]].mean(numeric_only=True).reset_index().sort_values(["task","macro_f1"], ascending=[True,False]))
else:
    print("Skipping Edge-IIoTset baselines because no Edge-IIoTset dataset was loaded.")

if RUN_E7_EDGE_CALIBRATION and edge_df is not None:
    e7_edge_df = run_edge_e7_calibration(edge_df, edge_label_col, OUTPUT_DIR)
    display(e7_edge_df[e7_edge_df["policy"].eq("none")].groupby(["model","calibration"])[["macro_f1","mcc","ece","brier_score","fa_10k"]].mean(numeric_only=True).reset_index().sort_values(["model","ece"]))
else:
    print("Skipping Edge-IIoTset calibration ECE comparison because no Edge-IIoTset dataset was loaded.")

if RUN_E8_EDGE_COMPRESSION and edge_df is not None:
    e8_edge_df = run_edge_e8_compression(edge_df, edge_label_col, OUTPUT_DIR)
    display(e8_edge_df.groupby("variant")[["macro_f1","mcc","model_size_mb","latency_per_flow_sec","throughput_flows_sec"]].mean(numeric_only=True).reset_index())
else:
    print("Skipping Edge-IIoTset compression trade-off comparison because no Edge-IIoTset dataset was loaded.")


## Result export


In [ ]:
# Generate manuscript-ready mean/std tables from E1-E8 CSVs

from pathlib import Path
import pandas as pd
import numpy as np
import re

# 1. Locate result files

RESULTS_DIR = Path(OUTPUT_DIR)
OUT_DIR = RESULTS_DIR / "paper_tables_final"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LATEX_DIR = OUT_DIR / "latex"
LATEX_DIR.mkdir(parents=True, exist_ok=True)

print("Reading from:", RESULTS_DIR)
print("Saving to:", OUT_DIR)

def find_latest_csv(patterns, search_dirs=None):
    if search_dirs is None:
        search_dirs = [RESULTS_DIR, PROJECT_ROOT, Path.cwd()]

    matches = []
    for d in search_dirs:
        if not d.exists():
            continue
        for pat in patterns:
            matches.extend(list(d.glob(pat)))

    if not matches:
        print(f"Warning: No file found for patterns: {patterns}")
        return None

    def version_score(p: Path):
        m = re.search(r"\((\d+)\)", p.name)
        v = int(m.group(1)) if m else 0
        return (p.stat().st_mtime, v)

    matches = sorted(set(matches), key=version_score, reverse=True)
    return matches[0]

FILES = {
    "E1": find_latest_csv(["E1_baselines*.csv"]),
    "E2": find_latest_csv(["E2_zero_day*.csv"]),
    "E3": find_latest_csv(["E3_compression*.csv"]),
    "E4": find_latest_csv(["E4_feature_ablation*.csv"]),
    "E5": find_latest_csv(["E5_calibration*.csv"]),
    "E6": find_latest_csv(["E6_edgeiiot_baselines*.csv"]),
    "E7": find_latest_csv(["E7_edgeiiot_calibration*.csv"]),
    "E8": find_latest_csv(["E8_edgeiiot_compression*.csv"]),
}

# 2. Load raw files safely

dfs = {k: (pd.read_csv(v) if v else pd.DataFrame()) for k, v in FILES.items()}

for k, df in dfs.items():
    if not df.empty:
        print(f"{k}: shape={df.shape}")
    else:
        print(f"{k}: Missing or empty")

# 3. Helper functions

METRIC_COLS = [
    "accuracy", "macro_f1", "weighted_f1", "mcc", "precision",
    "malicious_recall", "auprc", "auroc", "ece", "brier_score",
    "fa_10k", "false_positives", "benign_count", "model_size_mb",
    "latency_per_flow_sec", "throughput_flows_sec", "peak_rss_mb",
    "train_sec", "coverage",
]

PREFERRED_MAIN_METRICS = [
    "accuracy", "macro_f1", "mcc", "auprc", "auroc", "ece",
    "brier_score", "fa_10k", "model_size_mb", "latency_per_flow_sec",
    "throughput_flows_sec",
]

def existing(cols, df):
    return [c for c in cols if c in df.columns]

def mean_std_table(df, group_cols, metric_cols=None, round_digits=4):
    if metric_cols is None:
        metric_cols = existing(METRIC_COLS, df)
    else:
        metric_cols = existing(metric_cols, df)

    out = df.groupby(group_cols, dropna=False)[metric_cols].agg(["mean", "std"]).reset_index()

    flat_cols = []
    for col in out.columns:
        if isinstance(col, tuple):
            flat_cols.append(col[0] if col[1] == "" else f"{col[0]}_{col[1]}")
        else:
            flat_cols.append(col)
    out.columns = flat_cols

    for c in out.columns:
        if c not in group_cols and pd.api.types.is_numeric_dtype(out[c]):
            out[c] = out[c].round(round_digits)
    return out

def fmt_mean_std(mean, std, digits=4):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} $\\pm$ {std:.{digits}f}"

def formatted_table(mean_std_df, group_cols, metrics, digits=4):
    out = mean_std_df[group_cols].copy()
    for m in metrics:
        mean_col = f"{m}_mean"
        std_col = f"{m}_std"
        if mean_col in mean_std_df.columns:
            out[m] = [
                fmt_mean_std(a, b, digits=digits)
                for a, b in zip(
                    mean_std_df[mean_col],
                    mean_std_df[std_col] if std_col in mean_std_df.columns else [np.nan] * len(mean_std_df)
                )
            ]
    return out

def save_table(df, name, caption=None, label=None):
    csv_path = OUT_DIR / f"{name}.csv"
    tex_path = LATEX_DIR / f"{name}.tex"
    df.to_csv(csv_path, index=False)
    latex_str = df.to_latex(index=False, escape=False, caption=caption, label=label)
    tex_path.write_text(latex_str, encoding="utf-8")
    print(f"Saved: {csv_path}")
    return csv_path

def select_policy_none(df):
    out = df.copy()
    if "policy" in out.columns:
        out = out[out["policy"].astype(str).eq("none")]
    if "threshold" in out.columns:
        out = out[out["threshold"].isna()]
    return out

def best_by_metric(mean_std_df, group_cols, metric="macro_f1_mean"):
    df = mean_std_df.copy()
    idx = df.groupby(group_cols)[metric].idxmax()
    return df.loc[idx].reset_index(drop=True)

# Table Generators

excel_path = OUT_DIR / "all_manuscript_tables_mean_std.xlsx"
writer = pd.ExcelWriter(excel_path, engine="openpyxl")

if not dfs["E1"].empty:
    t2_numeric = mean_std_table(dfs["E1"], group_cols=["task", "model"], metric_cols=PREFERRED_MAIN_METRICS + ["train_sec"])
    t2 = formatted_table(t2_numeric, group_cols=["task", "model"], metrics=["accuracy", "macro_f1", "mcc", "auprc", "auroc", "ece", "fa_10k", "model_size_mb", "latency_per_flow_sec"])
    save_table(t2, "table_2_ciciot_baselines_mean_std", caption="CICIoT2023 baseline performance.", label="tab:baseline_results_ciciot")
    t2.to_excel(writer, sheet_name="T2_CICIoT_baselines", index=False)

    best_e1 = best_by_metric(t2_numeric, group_cols=["task"])
    best_e1_fmt = formatted_table(best_e1, group_cols=["task", "model"], metrics=["macro_f1", "mcc", "auprc", "auroc", "fa_10k"])
    save_table(best_e1_fmt, "compact_best_ciciot_model_per_task")
    best_e1_fmt.to_excel(writer, sheet_name="Best_CICIoT", index=False)

if not dfs["E2"].empty:
    t3_numeric = mean_std_table(dfs["E2"], group_cols=["heldout", "model"], metric_cols=["accuracy", "macro_f1", "mcc", "precision", "malicious_recall", "auprc", "auroc", "ece", "fa_10k"])
    t3 = formatted_table(t3_numeric, group_cols=["heldout", "model"], metrics=["macro_f1", "mcc", "malicious_recall", "auprc", "auroc", "fa_10k"])
    save_table(t3, "table_3_ciciot_zero_day_mean_std", caption="CICIoT2023 zero-day evaluation.", label="tab:zero_day_results")
    t3.to_excel(writer, sheet_name="T3_CICIoT_zero_day", index=False)

if not dfs["E6"].empty:
    t4_numeric = mean_std_table(dfs["E6"], group_cols=["task", "model"], metric_cols=PREFERRED_MAIN_METRICS + ["train_sec"])
    t4 = formatted_table(t4_numeric, group_cols=["task", "model"], metrics=["accuracy", "macro_f1", "mcc", "auprc", "auroc", "ece", "fa_10k", "model_size_mb", "latency_per_flow_sec"])
    save_table(t4, "table_4_edgeiiot_baselines_mean_std")
    t4.to_excel(writer, sheet_name="T4_Edge_baselines", index=False)

    best_e6 = best_by_metric(t4_numeric, group_cols=["task"])
    best_e6_fmt = formatted_table(best_e6, group_cols=["task", "model"], metrics=["macro_f1", "mcc", "auprc", "auroc", "fa_10k"])
    save_table(best_e6_fmt, "compact_best_edgeiiot_model_per_task")
    best_e6_fmt.to_excel(writer, sheet_name="Best_Edge", index=False)

comp_list = []
if not dfs["E3"].empty: comp_list.append(dfs["E3"].assign(dataset="CICIoT2023"))
if not dfs["E8"].empty: comp_list.append(dfs["E8"].assign(dataset="Edge-IIoTset"))
if comp_list:
    t5_numeric = mean_std_table(pd.concat(comp_list, ignore_index=True), group_cols=["dataset", "variant"], metric_cols=["accuracy", "macro_f1", "mcc", "auprc", "auroc", "ece", "fa_10k", "model_size_mb", "latency_per_flow_sec", "throughput_flows_sec"])
    t5 = formatted_table(t5_numeric, group_cols=["dataset", "variant"], metrics=["macro_f1", "mcc", "model_size_mb", "latency_per_flow_sec", "throughput_flows_sec", "fa_10k"])
    save_table(t5, "table_5_compression_tradeoff_mean_std")
    t5.to_excel(writer, sheet_name="T5_Compression", index=False)

calib_list = []
if not dfs["E5"].empty: calib_list.append(dfs["E5"].assign(dataset="CICIoT2023"))
if not dfs["E7"].empty: calib_list.append(dfs["E7"].assign(dataset="Edge-IIoTset"))
if calib_list:
    t6_numeric = mean_std_table(select_policy_none(pd.concat(calib_list, ignore_index=True)), group_cols=["dataset", "model", "calibration"], metric_cols=["accuracy", "macro_f1", "mcc", "auprc", "auroc", "ece", "brier_score", "fa_10k"])
    t6 = formatted_table(t6_numeric, group_cols=["dataset", "model", "calibration"], metrics=["macro_f1", "mcc", "ece", "brier_score", "fa_10k"])
    save_table(t6, "table_6_calibration_reliability_mean_std")
    t6.to_excel(writer, sheet_name="T6_Calibration", index=False)

if not dfs["E4"].empty:
    t7_numeric = mean_std_table(dfs["E4"], group_cols=["feature_group", "model"], metric_cols=["accuracy", "macro_f1", "mcc", "auprc", "auroc", "ece", "fa_10k", "n_features"])
    t7 = formatted_table(t7_numeric, group_cols=["feature_group", "model"], metrics=["macro_f1", "mcc", "auprc", "auroc", "fa_10k"])
    if "n_features_mean" in t7_numeric.columns:
        t7.insert(2, "n_features", t7_numeric["n_features_mean"].round(0).astype("Int64").astype(str))
    save_table(t7, "table_7_ciciot_feature_ablation_mean_std")
    t7.to_excel(writer, sheet_name="T7_Ablation", index=False)

def threshold_policy_table(df, dataset_name):
    if df.empty or "policy" not in df.columns: return None
    out = df[df["policy"].astype(str).eq("conservative_no_alert")].copy()
    if out.empty: return None
    numeric = mean_std_table(out, group_cols=["model", "calibration", "threshold"], metric_cols=["accuracy", "macro_f1", "mcc", "ece", "brier_score", "fa_10k", "coverage", "malicious_recall"])
    formatted = formatted_table(numeric, group_cols=["model", "calibration", "threshold"], metrics=["macro_f1", "mcc", "ece", "fa_10k", "coverage", "malicious_recall"])
    save_table(formatted, f"supp_threshold_policy_{dataset_name.lower().replace('-', '').replace(' ', '_')}")
    return formatted

supp_ciciot = threshold_policy_table(dfs["E5"], "CICIoT2023")
if supp_ciciot is not None: supp_ciciot.to_excel(writer, sheet_name="Supp_CICIoT_threshold", index=False)

supp_edge = threshold_policy_table(dfs["E7"], "Edge-IIoTset")
if supp_edge is not None: supp_edge.to_excel(writer, sheet_name="Supp_Edge_threshold", index=False)

writer.close()
print("\nDone. Excel workbook saved to:", excel_path)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", font_scale=1.2)

RESULTS_DIR = Path(OUTPUT_DIR)
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    "E1": RESULTS_DIR / "E1_baselines.csv",
    "E2": RESULTS_DIR / "E2_zero_day.csv",
    "E3": RESULTS_DIR / "E3_compression.csv",
    "E4": RESULTS_DIR / "E4_feature_ablation.csv",
    "E5": RESULTS_DIR / "E5_calibration.csv",
    "E6_edge": RESULTS_DIR / "E6_edgeiiot_baselines.csv",
    "E7_edge": RESULTS_DIR / "E7_edgeiiot_calibration.csv",
    "E8_edge": RESULTS_DIR / "E8_edgeiiot_compression.csv",
}

results = {k: (pd.read_csv(v) if v.exists() else pd.DataFrame()) for k, v in RAW_FILES.items()}

MODEL_MAP = {
    "lr": "Logistic Regression",
    "rf": "Random Forest",
    "lgbm": "LightGBM",
    "tinymlp": "TinyMLP",
}

TASK_MAP = {
    "binary": "Binary Classification",
    "multiclass": "Multiclass Classification",
}

e6_df = results.get("E6", results.get("E6_edge", pd.DataFrame()))

if results["E1"].empty or e6_df.empty:
    print("Skipping dataset validation figure export because E1 or E6 result files are missing.")
else:
    val = pd.concat([
        results["E1"].assign(dataset="CICIoT2023")[["dataset", "task", "model", "macro_f1"]],
        e6_df.assign(dataset="Edge-IIoTset")[["dataset", "task", "model", "macro_f1"]],
    ], ignore_index=True)

    val["model"] = val["model"].replace(MODEL_MAP)
    val["task"] = val["task"].replace(TASK_MAP)

    val_summary = (
        val.groupby(["dataset", "task", "model"], dropna=False)["macro_f1"]
           .agg(["mean", "std"])
           .reset_index()
           .rename(columns={"mean": "macro_f1_mean", "std": "macro_f1_std"})
    )
    val_summary["macro_f1_mean"] = val_summary["macro_f1_mean"].round(4)
    val_summary.to_csv(FIG_DIR / "dataset_independent_validation_macro_f1.csv", index=False)

    for task in val_summary["task"].unique():
        sub = val_summary[val_summary["task"].eq(task)].copy()

        plt.figure(figsize=(10, 6))
        ax = sns.barplot(data=sub, x="dataset", y="macro_f1_mean", hue="model")

        for container in ax.containers:
            ax.bar_label(container, fmt="%.4f", padding=3, fontsize=10)

        plt.ylabel("Macro-F1")
        plt.xlabel("Dataset")
        plt.ylim(0, 1.05)
        plt.legend(title="Model", loc="best")
        plt.tight_layout()

        safe_task = task.lower().replace(" ", "_")
        png = FIG_DIR / f"validation_{safe_task}.png"
        eps = FIG_DIR / f"validation_{safe_task}.eps"
        csv = FIG_DIR / f"validation_{safe_task}.csv"

        sub.to_csv(csv, index=False)
        plt.savefig(png, dpi=300)
        plt.savefig(eps, format="eps")
        plt.show()

        print("Saved:", png)
        print("Saved:", eps)
        print("Saved:", csv)


In [ ]:
import pandas as pd
from pathlib import Path

MODEL_MAP = {"lr": "Logistic Regression", "rf": "Random Forest", "lgbm": "LightGBM", "tinymlp": "TinyMLP"}
CALIB_MAP = {"uncalibrated": "Uncalibrated", "platt_sigmoid": "Platt Sigmoid", "isotonic": "Isotonic Regression", "temperature": "Temperature Scaling"}
VARIANT_MAP = {"fp32_gpu": "FP32 (GPU)", "pruned30_fp32": "Pruned 30% FP32", "int8_dynamic_cpu": "INT8 Dynamic (CPU)", "pruned30_int8_cpu": "Pruned 30% INT8 (CPU)"}
TASK_MAP = {"binary": "Binary Classification", "multiclass": "Multiclass Classification"}

# 1. CICIoT2023 compression trade-off data
if not results["E3"].empty:
    df_e3 = results["E3"].copy()
    df_e3["variant"] = df_e3["variant"].replace(VARIANT_MAP)
    df_e3_agg = df_e3.groupby("variant")[["macro_f1","model_size_mb","latency_per_flow_sec","throughput_flows_sec"]].mean(numeric_only=True).reset_index()
    df_e3_agg.to_csv(FIG_DIR / "ciciot_compression_size_vs_macro_f1.csv", index=False)

# 2. CICIoT2023 calibration ECE bar data
if not results["E5"].empty:
    df_e5 = results["E5"][results["E5"]["policy"].fillna("none").eq("none")].copy()
    df_e5["model"] = df_e5["model"].replace(MODEL_MAP)
    df_e5["calibration"] = df_e5["calibration"].replace(CALIB_MAP)
    df_e5_agg = df_e5.groupby(["model","calibration"])[["ece","brier_score","macro_f1"]].mean(numeric_only=True).reset_index()
    df_e5_agg.to_csv(FIG_DIR / "ciciot_calibration_ece_comparison.csv", index=False)
    # Also save the grouped bar/boxplot source data (raw results for CIs)
    df_e5.to_csv(FIG_DIR / "ciciot_calibration_ece_grouped_bar.csv", index=False)

# 3. Edge-IIoTset compression trade-off data
if not results["E8_edge"].empty:
    df_e8 = results["E8_edge"].copy()
    df_e8["variant"] = df_e8["variant"].replace(VARIANT_MAP)
    df_e8_agg = df_e8.groupby("variant")[["macro_f1","model_size_mb","latency_per_flow_sec","throughput_flows_sec"]].mean(numeric_only=True).reset_index()
    df_e8_agg.to_csv(FIG_DIR / "edgeiiot_compression_size_vs_macro_f1.csv", index=False)

# 4. Edge-IIoTset calibration data (Heatmap and Bar/Radar)
if not results["E7_edge"].empty:
    df_e7 = results["E7_edge"][results["E7_edge"]["policy"].fillna("none").eq("none")].copy()
    df_e7["model"] = df_e7["model"].replace(MODEL_MAP)
    df_e7["calibration"] = df_e7["calibration"].replace(CALIB_MAP)
    df_e7_agg = df_e7.groupby(["model","calibration"])[["ece","brier_score","macro_f1"]].mean(numeric_only=True).reset_index()
    df_e7_agg.to_csv(FIG_DIR / "edgeiiot_calibration_ece_comparison.csv", index=False)
    df_e7_agg.to_csv(FIG_DIR / "edgeiiot_calibration_ece_heatmap.csv", index=False)
    df_e7_agg.to_csv(FIG_DIR / "edgeiiot_calibration_ece_radar.csv", index=False)
    df_e7.to_csv(FIG_DIR / "edgeiiot_calibration_ece_grouped_bar.csv", index=False)

# 5. Dataset-independent validation summary data
if not results["E1"].empty and not results["E6_edge"].empty:
    c1_tmp = results["E1"].assign(dataset="CICIoT2023")[["dataset","task","model","macro_f1","mcc"]].copy()
    e1_tmp = results["E6_edge"].assign(dataset="Edge-IIoTset")[["dataset","task","model","macro_f1","mcc"]].copy()
    comp_tmp = pd.concat([c1_tmp, e1_tmp], ignore_index=True)
    comp_tmp["model"] = comp_tmp["model"].replace(MODEL_MAP)
    comp_tmp["task"] = comp_tmp["task"].replace(TASK_MAP)
    summary_tmp = comp_tmp.groupby(["dataset","task","model"])["macro_f1"].mean(numeric_only=True).reset_index()
    for t in summary_tmp["task"].unique():
        t_clean = t.lower().replace(" ", "_")
        summary_tmp[summary_tmp["task"] == t].to_csv(FIG_DIR / f"dataset_independent_validation_{t_clean}_macro_f1.csv", index=False)

print(f"Figure source CSV files saved to: {FIG_DIR}")


In [ ]:
# Final ECE comparison figures
# Source: E5_calibration.csv + E7_edgeiiot_calibration.csv

CAL_MAP = {
    "uncalibrated": "Uncalibrated",
    "isotonic": "Isotonic",
    "platt_sigmoid": "Platt Sigmoid",
    "temperature": "Temperature",
    "Isotonic": "Isotonic",
    "Platt sigmoid": "Platt Sigmoid",
    "Temperature": "Temperature",
    "Uncalibrated": "Uncalibrated",
}

def base_calibration_rows(df):
    out = df.copy()
    if "policy" in out.columns:
        out = out[out["policy"].fillna("none").eq("none")]
    if "threshold" in out.columns:
        out = out[out["threshold"].isna()]
    out["model"] = out["model"].replace(MODEL_MAP)
    out["calibration"] = out["calibration"].replace(CAL_MAP)
    return out

for dataset_name, key, out_prefix in [
    ("CICIoT2023", "E5", "ciciot_calibration_ece_grouped_bar"),
    ("Edge-IIoTset", "E7_edge", "edgeiiot_calibration_ece_grouped_bar"),
]:
    actual_key = key if key in results else key.replace('_edge', '')
    if actual_key not in results or results[actual_key].empty:
        print(f"Skipping {dataset_name} ECE figures because {actual_key} is missing or empty.")
        continue

    base = base_calibration_rows(results[actual_key])

    # Save exact source data used for plotting.
    base.to_csv(FIG_DIR / f"{out_prefix}.csv", index=False)

    plt.figure(figsize=(10, 5))
    sns.set_theme(style="whitegrid", font_scale=1.2)

    ax = sns.barplot(
        data=base,
        x="calibration",
        y="ece",
        hue="model",
        errorbar=("ci", 95),
        capsize=0.1,
    )

    for container in ax.containers:
        ax.bar_label(container, fmt="%.4f", padding=7, fontsize=9)

    # plt.title(f"{dataset_name}: ECE Comparison")
    plt.ylabel("Expected Calibration Error (ECE)")
    plt.xlabel("Calibration Strategy")
    plt.legend(title="Model", loc="best")
    plt.tight_layout()

    png = FIG_DIR / f"{out_prefix}.png"
    eps = FIG_DIR / f"{out_prefix}.eps"

    plt.savefig(png, dpi=300)
    plt.savefig(eps, format="eps")
    plt.show()

    print("Saved:", png)
    print("Saved:", eps)


In [ ]:
# Consistency checks between final tables and figure source data.
required_objects = ["t2_numeric", "t4_numeric", "t6_numeric"]
if not all(name in globals() for name in required_objects):
    print("Skipping consistency checks because the required summary tables were not generated.")
else:
    # Check validation figure values against baseline mean+/-std table sources.
    val_check = pd.concat([
        t2_numeric.assign(dataset="CICIoT2023"),
        t4_numeric.assign(dataset="Edge-IIoTset"),
    ], ignore_index=True)

    val_check = val_check[["dataset", "task", "model", "macro_f1_mean"]].copy()
    val_check["model"] = val_check["model"].replace(MODEL_MAP)
    val_check["task"] = val_check["task"].replace(TASK_MAP)

    fig_val = pd.read_csv(FIG_DIR / "dataset_independent_validation_macro_f1.csv")
    merged = val_check.merge(
        fig_val[["dataset", "task", "model", "macro_f1_mean"]],
        on=["dataset", "task", "model"],
        suffixes=("_table", "_figure"),
    )

    merged["abs_diff"] = (merged["macro_f1_mean_table"] - merged["macro_f1_mean_figure"]).abs()
    print(merged)

    if (merged["abs_diff"] > 1e-10).any():
        raise ValueError("Validation figure values do not match table source values.")

    # Check calibration figure model set against calibration table source.
    for dataset_name, fig_file in [
        ("CICIoT2023", "ciciot_calibration_ece_grouped_bar.csv"),
        ("Edge-IIoTset", "edgeiiot_calibration_ece_grouped_bar.csv"),
    ]:
        fig_df = pd.read_csv(FIG_DIR / fig_file)
        table_models = set(
            t6_numeric[t6_numeric["dataset"].eq(dataset_name)]["model"].replace(MODEL_MAP)
        )
        fig_models = set(fig_df["model"])
        print(dataset_name, "table models:", table_models, "figure models:", fig_models)
        if table_models != fig_models:
            raise ValueError(f"{dataset_name}: calibration figure models do not match calibration table models.")

    print("All table/figure consistency checks passed.")
